# NumPy ufuncs on rate expressions

Seasonal forcing needs `sin`. Until this page, the only non-operator
functions on a rate were eight named helpers (`exp`, `log`, `tanh`,
`sqrt`, `floor`, `maximum`, `minimum`, `clip`) — `sin` was not one of
them, and `jnp.sin` cannot see a rate expression at all.

The claim: `np.sin(Time())` is a rate. Over one year, an entry rate of
`0.1 * (1 + 0.5 * sin(2π t / 365))` drawn from the model lies on top of
the same formula evaluated in NumPy. `jnp.sin` of that same expression
still raises.


## Seasonal contact over a year

The next cell builds the rate with `np.sin`, evaluates it as an entry flow
at each day, and plots it against NumPy. The two lines should overlie.
It also shows that `jnp.sin` is refused.


In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import jax.numpy as jnp

from summer4 import EntryFlow, FlowModel, Param, Property, PropertyData, PropertyMap, Time

state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
rate = 0.1 * (1.0 + 0.5 * np.sin(2.0 * np.pi * Time() / 365.0))
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["Y"], rate))
compiled = model.compile()
y0 = PropertyData.wrap(pmap, np.array([0.0]))

days = np.linspace(0.0, 365.0, 366)


def model_rate(t: float) -> float:
    return float(np.asarray(compiled.vector_field(t, y0, {}).data)[0])


modelled = np.array([model_rate(float(t)) for t in days])
formula = 0.1 * (1.0 + 0.5 * np.sin(2.0 * np.pi * days / 365.0))
np.testing.assert_allclose(modelled, formula, rtol=1e-5, atol=1e-5)

try:
    jnp.sin(Param("phase"))
except TypeError:
    jnp_refused = True
else:
    jnp_refused = False
assert jnp_refused, "jnp.sin must refuse a rate expression; JAX has no dispatch hook"

pd.DataFrame({"model": modelled, "numpy sin": formula}, index=days).plot(
    title="Seasonal entry rate from np.sin(Time()) matches NumPy",
    labels={"index": "day", "value": "people per day"},
)
